# 14 · Inference and Model Serving

In plain English, **inference** is the moment you actually *use* a model — you give it an input and it gives you an answer. All the training you've done in earlier notebooks was about *making* the model good; this notebook is about *putting it to work*. We'll generate text, classify leads, load a LoRA adapter, save and reload everything, speed things up with batching, and finally wrap a model in a tiny web API so a real product could call it.

Everything here runs on a plain laptop **CPU** with **small, free models** (`distilgpt2` for generation, a tiny DistilBERT-style model for classification). No GPU required.

## What you'll learn

- **Inference vs. training**: why inference is just the **forward pass**, and why we use `model.eval()` + `torch.no_grad()`.
- **Text generation** with `distilgpt2`: `model.generate(...)` and the decoding knobs — `max_new_tokens`, `do_sample`, `temperature`, `top_k`, `top_p` — plus **greedy vs. sampling** on the same prompt.
- **Classifier inference**: forward pass → `softmax` → `argmax` → map the id back to a label (`hot` / `warm` / `cold`), wrapped in a tidy `predict(text)` helper.
- **Loading a LoRA adapter** for inference with `PeftModel.from_pretrained(...)`, and `merge_and_unload()` to fold it into the base model.
- **Saving & loading** with `save_pretrained` / `from_pretrained` — and why you must ship the tokenizer too.
- **Batched inference** for speed: tokenize a list with `padding=True` and run it all at once.
- **Serving behind an API**: a minimal **FastAPI** app you save as `app.py` and run with `uvicorn`.
- A short note on making inference **cheaper** (quantization, smaller models, caching).

## Why this matters for fine-tuning

Fine-tuning is only half the job. A fine-tuned model sitting on your disk does nothing — it has to be *served* somewhere so your application can send it inputs and get predictions back. This notebook is the bridge between "I trained a model" and "my product uses a model."

Concretely, in the capstone (notebook 15) you'll fine-tune a small model to score incoming sales **leads** as `hot`, `warm`, or `cold`. Everything you learn here — loading the saved model, running a clean forward pass, mapping ids to labels, merging a LoRA adapter, batching for speed, and exposing a `/predict` endpoint — is exactly how that fine-tuned lead-scorer would run inside a real CRM or marketing tool. Inference and serving are how your training effort finally pays off.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it on Google Colab or a fresh environment.

We also detect the best available **device** (`cuda` GPU, Apple `mps`, or `cpu`). Everything in this notebook is small enough to run on `cpu`, but if you have a GPU it'll be used automatically.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install transformers torch peft

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)

# Pick the best device available. For these tiny models, CPU is totally fine.
if torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon GPU
else:
    device = torch.device("cpu")        # plain CPU (works everywhere)

print("Using device:", device)
print("torch version:", torch.__version__)

## 1. Inference vs. training: just the forward pass

Training is a loop of four steps: **forward pass** (compute a prediction), **compute loss** (how wrong it was), **backward pass** (compute gradients), and **update weights**. Inference throws away the last three. You only ever do the **forward pass** — input goes in, prediction comes out, nothing about the model changes.

Because of that, you should always wrap inference in two things:

- **`model.eval()`** — switches the model into "evaluation mode." Some layers (like *dropout* and *batch-norm*) behave differently during training vs. when making real predictions. `eval()` tells them "we're predicting now, behave deterministically."
- **`torch.no_grad()`** — tells PyTorch "don't track gradients." Gradients are only needed for the backward pass, which we're *not* doing. Skipping them makes inference **faster and lighter on memory**.

Forgetting either one won't usually crash, but it wastes memory and can subtly change your outputs. Make them a habit.

In [ ]:
# A mental model of the two modes. (No real model here — just the pattern.)

# --- TRAINING (what earlier notebooks did) ---
# model.train()                 # training mode (dropout ON, etc.)
# outputs = model(**inputs)     # 1) forward pass
# loss = outputs.loss           # 2) measure error
# loss.backward()               # 3) backward pass (gradients)
# optimizer.step()              # 4) update weights
# optimizer.zero_grad()

# --- INFERENCE (this whole notebook) ---
# model.eval()                  # evaluation mode (dropout OFF)
# with torch.no_grad():         # no gradient tracking -> faster, less memory
#     outputs = model(**inputs) # JUST the forward pass. Done.

print("Inference = forward pass only. Use model.eval() + torch.no_grad().")

**What this does:** This cell is a **side-by-side reference**, not runnable code — it contrasts the 4-step training loop with the 1-step inference path. At inference time you never call `.backward()`, never touch an optimizer, and the weights stay frozen — you only read the model's output. You'll see `model.eval()` and `with torch.no_grad():` in almost every cell from here on.

### ✏️ Exercise

In your own words (in a comment), explain *why* we don't need `loss.backward()` during inference. What is a gradient used for, and do we ever update weights when we're just making predictions?

## 2. Text generation with `distilgpt2`

A **causal language model** (like GPT) generates text by predicting the next token over and over: it looks at everything so far, picks a next token, appends it, and repeats. We'll use **`distilgpt2`** — a tiny, distilled GPT-2 that runs fine on CPU. It's not smart (it's small!), but it's perfect for learning the *mechanics* of generation.

First, load the model and tokenizer and move the model to our `device`.

In [ ]:
gen_name = "distilgpt2"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_name)

gen_model.to(device)   # move model weights onto our chosen device
gen_model.eval()       # inference mode

# GPT-2 has no dedicated padding token. We reuse the end-of-text token as pad.
# This avoids warnings/errors when we batch or generate later.
gen_tokenizer.pad_token = gen_tokenizer.eos_token

print("Loaded", gen_name)
print("eos_token_id:", gen_tokenizer.eos_token_id,
      "| pad_token_id:", gen_tokenizer.pad_token_id)

**What this does:**

- `AutoModelForCausalLM` loads a **generation** model (the GPT family), as opposed to a classification model.
- `.to(device)` moves the weights to CPU/GPU; `.eval()` sets inference mode.
- `gen_tokenizer.pad_token = gen_tokenizer.eos_token`: GPT-2 wasn't trained with a padding token, so we tell it "use the **end-of-text** token as padding." We'll need padding when we batch multiple prompts together later.
- **`eos_token_id`** marks the end of a sequence; **`pad_token_id`** is the filler used to make sequences equal length. Generation uses both to know when to stop and how to handle padding.

## 3. Greedy decoding: the "most likely" path

The simplest way to generate is **greedy decoding**: at every step, pick the single **most likely** next token. It's fast and **deterministic** — the same prompt always gives the same output. The downside: it can be repetitive and a bit boring, because it never takes a chance.

The key knob is **`max_new_tokens`** — how many *new* tokens to generate after your prompt. Keep it small (like 30) so CPU runs stay quick.

In [ ]:
prompt = "The best way to learn machine learning is"

# Tokenize the prompt and move the tensors to the device.
inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)

# Greedy decoding = do_sample=False (the default).
with torch.no_grad():
    greedy_ids = gen_model.generate(
        **inputs,
        max_new_tokens=30,        # add up to 30 new tokens
        do_sample=False,          # greedy: always take the most likely token
        pad_token_id=gen_tokenizer.pad_token_id,  # silence a harmless warning
    )

greedy_text = gen_tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
print("GREEDY:\n", greedy_text)

# Run this cell twice — you'll get the EXACT same text both times.

**What this does:**

- `gen_tokenizer(prompt, return_tensors="pt")` turns the prompt into `input_ids` tensors; `.to(device)` moves them next to the model.
- `gen_model.generate(...)` runs the next-token loop for you — you don't write the loop by hand.
- `do_sample=False` is **greedy**: each step takes the highest-probability token. That's why it's **deterministic** (same output every run).
- `max_new_tokens=30` caps the length. `pad_token_id=...` just avoids a harmless warning.
- `tokenizer.decode(..., skip_special_tokens=True)` turns the output ids back into readable text and hides markers like `<|endoftext|>`.

### ✏️ Exercise

Run the greedy cell with `max_new_tokens=10` and again with `max_new_tokens=60`. The shorter one finishes faster. Then change the `prompt` to something of your own and confirm the output is still identical across two runs (because greedy is deterministic).

In [ ]:
# Your turn:
# inputs = gen_tokenizer("Once upon a time", return_tensors="pt").to(device)
# with torch.no_grad():
#     out = gen_model.generate(**inputs, max_new_tokens=10, do_sample=False,
#                              pad_token_id=gen_tokenizer.pad_token_id)
# print(gen_tokenizer.decode(out[0], skip_special_tokens=True))

## 4. Sampling: adding controlled randomness

**Sampling** (`do_sample=True`) makes generation more creative by *rolling dice* among the likely next tokens instead of always taking the top one. The same prompt now gives **different** outputs each run. Three knobs control how wild it gets:

- **`temperature`** — the "creativity dial." Low (e.g. `0.2`) = cautious, sticks to safe choices. High (e.g. `1.2`) = adventurous, more surprising (and more likely to ramble). `1.0` is neutral.
- **`top_k`** — only consider the **k** most likely tokens at each step (e.g. `top_k=50` ignores everything outside the top 50). Blocks truly unlikely words.
- **`top_p`** (nucleus sampling) — consider just enough top tokens for their probabilities to add up to **p** (e.g. `top_p=0.9`). Adapts the pool size to how confident the model is.

You typically use `top_k` and `top_p` together to keep output sensible while still varied.

In [ ]:
inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)

# Sampling: do_sample=True plus the creativity knobs.
torch.manual_seed(0)  # seed just so this cell is reproducible in the lesson
with torch.no_grad():
    sampled_ids = gen_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,      # turn ON sampling (randomness)
        temperature=0.9,     # mild creativity
        top_k=50,            # only sample from the 50 most likely tokens
        top_p=0.95,          # ...whose probabilities sum to <= 0.95
        pad_token_id=gen_tokenizer.pad_token_id,
    )

sampled_text = gen_tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
print("SAMPLED:\n", sampled_text)

# Remove the torch.manual_seed line above and re-run: the output CHANGES each time.

**What this does:**

- `do_sample=True` switches from "always pick the top token" to "randomly pick from the likely tokens," so output varies run to run.
- `temperature=0.9` keeps creativity mild; `top_k=50` and `top_p=0.95` restrict choices to sensible tokens so it doesn't produce gibberish.
- We set `torch.manual_seed(0)` *only* to make this teaching example reproducible. In real use you'd leave it out and embrace the variety — or set a seed when you need repeatable results (e.g. tests).

### Same prompt, two strategies, side by side

Let's print **greedy vs. sampled** for the exact same prompt so the difference is obvious. **When to use which?** Use **greedy** (or low temperature) when you want consistent, predictable answers — e.g. extracting a label or a factual reply. Use **sampling** when you want variety or creativity — e.g. brainstorming or story text. The same choice applies to a *fine-tuned* generation model.

In [ ]:
print("PROMPT:", prompt)
print("\n--- GREEDY (deterministic, 'safest' path) ---")
print(greedy_text)
print("\n--- SAMPLED (random, more varied) ---")
print(sampled_text)

# Key idea:
# * Greedy  -> same output every time. Good for factual/consistent tasks.
# * Sampling -> different each time. Good for creative/varied text.

**What this does:** Side by side you can see the **greedy** output is the model's single "most likely" continuation, while the **sampled** one took some chances and reads differently.

### ✏️ Exercise

Generate with the same prompt using `temperature=0.2` and then `temperature=1.5` (keep `do_sample=True`). Read both. The low-temperature output should feel "safe" and repetitive; the high one should feel wilder and may go off the rails. Write a one-line comment describing the difference you saw.

In [ ]:
# Your turn:
# for t in [0.2, 1.5]:
#     with torch.no_grad():
#         out = gen_model.generate(**inputs, max_new_tokens=30, do_sample=True,
#                                  temperature=t, top_k=50, top_p=0.95,
#                                  pad_token_id=gen_tokenizer.pad_token_id)
#     print(f"temperature={t}:", gen_tokenizer.decode(out[0], skip_special_tokens=True), "\n")
# Difference I noticed: ...

## 5. Classifier inference: scoring a lead

Generation is one kind of model; **classification** is the other big one — and it's what the capstone lead-scorer is. A classifier takes text and outputs a **label**. For leads, the labels are **`hot`**, **`warm`**, and **`cold`**.

The recipe is always the same three steps you saw in notebook 08:

1. **Forward pass** → raw scores called **logits** (one per class).
2. **`softmax`** → turn logits into **probabilities** that sum to 1.
3. **`argmax`** → the index of the biggest probability = the **predicted class id**, which we map back to a label.

We'll use a tiny untrained classifier here just to demonstrate the mechanics. (In the capstone this model will be your *fine-tuned* one — the code below stays identical; only the weights change.)

In [ ]:
# A deliberately tiny model so it loads fast on CPU. We tell it there are
# 3 classes (hot/warm/cold). Since it's NOT fine-tuned yet, predictions are
# random — we only care about the INFERENCE MECHANICS here.
clf_name = "prajjwal1/bert-tiny"

# Our label maps. In a fine-tuned model these are saved in model.config.
id2label = {0: "hot", 1: "warm", 2: "cold"}
label2id = {v: k for k, v in id2label.items()}

clf_tokenizer = AutoTokenizer.from_pretrained(clf_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(
    clf_name,
    num_labels=3,
    id2label=id2label,   # so the model carries our label names
    label2id=label2id,
)
clf_model.to(device)
clf_model.eval()

print("Classifier ready. id2label =", clf_model.config.id2label)

**What this does:**

- `AutoModelForSequenceClassification(..., num_labels=3)` puts a fresh 3-way classification head on a tiny BERT — one output per class (`hot`, `warm`, `cold`).
- Passing `id2label` / `label2id` makes the model **carry its own label names**, so we never have to remember "is class 0 hot or cold?" — it's stored in `model.config`.
- We move it to `device` and set `eval()`. It's untrained, so don't trust its answers yet — we're learning the *plumbing*.

In [ ]:
# One forward pass, step by step.
lead_text = "Requested a demo and asked about enterprise pricing today."

inputs = clf_tokenizer(lead_text, return_tensors="pt").to(device)

with torch.no_grad():                       # inference: no gradients
    logits = clf_model(**inputs).logits     # 1) raw scores, shape [1, 3]

probs = torch.softmax(logits, dim=-1)       # 2) -> probabilities summing to 1
pred_id = torch.argmax(probs, dim=-1).item()# 3) index of the largest probability

print("logits:       ", logits)
print("probabilities:", probs)
print("predicted id: ", pred_id)
print("PREDICTED LABEL:", clf_model.config.id2label[pred_id])
# (Label is random for now because the model isn't fine-tuned.)

**What this does:** `clf_model(**inputs).logits` is the forward pass — three raw scores, one per class. `torch.softmax(..., dim=-1)` turns them into three probabilities that sum to 1.0; `torch.argmax(...).item()` finds which class won as a plain integer; `model.config.id2label[pred_id]` turns it into `"hot"`/`"warm"`/`"cold"`. This **logits → softmax → argmax → label** chain is how *every* classifier answers.

### Wrap it in a `predict(text)` helper

In real code you don't want to repeat those steps everywhere. Bundle them into one small function that takes raw text and returns a clean result — the label plus the confidence. This is exactly the helper you'd call from an API endpoint later.

In [ ]:
def predict(text):
    """Run the classifier on one string and return (label, confidence)."""
    inputs = clf_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    pred_id = torch.argmax(probs, dim=-1).item()
    label = clf_model.config.id2label[pred_id]
    confidence = probs[0, pred_id].item()        # probability of the winning class
    return {"label": label, "confidence": round(confidence, 3)}

# Try it:
print(predict("Just browsing, not ready to buy anything yet."))
print(predict("Wants to sign the contract this week, budget approved."))

**What this does:**

- `predict(text)` packs tokenize → forward pass → softmax → argmax → label into one call and returns a small dict like `{"label": "warm", "confidence": 0.41}`.
- `confidence` is the probability of the winning class — useful for deciding whether to trust the prediction or route it to a human.
- This is the **core function** your capstone API will expose. Swap in the fine-tuned model and the helper instantly gives meaningful labels.

### ✏️ Exercise

Call `predict(...)` on three lead descriptions of your own — one that *sounds* hot, one warm, one cold. Because this model isn't fine-tuned, the labels will be random; that's expected. The point is to confirm the helper returns a clean `{"label": ..., "confidence": ...}` dict every time without errors.

In [ ]:
# Your turn:
# print(predict("..."))
# print(predict("..."))
# print(predict("..."))

## 6. Loading a LoRA adapter for inference

In the LoRA/QLoRA notebooks you fine-tuned a model **cheaply** by training only a tiny set of extra weights called an **adapter**, leaving the big base model frozen. After training, LoRA saves *just the adapter* (a few megabytes) — not a full copy of the model.

So to *use* a LoRA-fine-tuned model at inference time you do two things:

1. Load the **original base model** (the big frozen one).
2. **Attach the adapter** on top with `PeftModel.from_pretrained(base_model, adapter_path)`.

The cell below is **illustrative** (commented out) because it needs the `peft` library and an adapter folder you'd have saved during the LoRA notebook. Read it as the pattern you'll follow.

In [ ]:
# Pattern for loading a LoRA adapter (requires: pip install peft)
#
# from peft import PeftModel
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
#
# base_name = "distilbert-base-uncased"   # the ORIGINAL base model
# adapter_path = "my_lora_adapter"        # folder saved during LoRA training
#
# # 1) Load the frozen base model (3 classes for our lead task).
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     base_name, num_labels=3
# )
#
# # 2) Attach the trained adapter on top of the base.
# lora_model = PeftModel.from_pretrained(base_model, adapter_path)
# lora_model.eval()
#
# # Now lora_model behaves like a normal model for inference:
# # tokenizer = AutoTokenizer.from_pretrained(base_name)
# # inputs = tokenizer("a lead description", return_tensors="pt")
# # with torch.no_grad():
# #     logits = lora_model(**inputs).logits

print("PeftModel.from_pretrained(base_model, adapter_path) = base + adapter.")

**What this does:** `PeftModel.from_pretrained(base_model, adapter_path)` doesn't load a whole new model — it loads the **small adapter** and layers it onto the base you already have in memory. The result acts like any other model (same `eval()` + `torch.no_grad()` + forward-pass recipe). This is why LoRA is storage-friendly: you keep **one** copy of the big base and many tiny adapters (one per task), swapping them in as needed.

### Merge the adapter into the base: `merge_and_unload()`

Keeping the base and adapter *separate* is flexible — you can swap adapters at runtime. But for a simple deployment it's often easier to **bake the adapter into the base model** so you have a single, standalone model with no `peft` dependency. That's what `merge_and_unload()` does: it folds the adapter's weights into the base and gives you back a plain model.

In [ ]:
# Pattern for merging (also requires peft + an adapter):
#
# merged_model = lora_model.merge_and_unload()   # fold adapter into base
# merged_model.save_pretrained("merged_lead_model")
# tokenizer.save_pretrained("merged_lead_model")
#
# # "merged_lead_model" is now a normal model folder. Load it WITHOUT peft:
# # from transformers import AutoModelForSequenceClassification
# # model = AutoModelForSequenceClassification.from_pretrained("merged_lead_model")

print("merge_and_unload() -> one standalone model, no adapter needed at runtime.")

**What this does — and when to merge vs. keep separate:**

- `merge_and_unload()` permanently combines adapter + base into a single set of weights, then removes the `peft` wrapper. Save it and you get an ordinary model folder anyone can load with plain `transformers`.

**Merge** (simpler deploy) when:
- You serve **one** fine-tuned task and want the smallest, simplest runtime (no `peft` install, slightly faster).

**Keep separate** (don't merge) when:
- You want to **swap adapters** at runtime (e.g. one adapter for lead-scoring, another for support-ticket tagging) while sharing a single base model in memory — saves a lot of RAM/disk.

### ✏️ Exercise

No code to run (it needs a real adapter). Decide: if your product must serve **five different fine-tuned tasks** that all share the same base model on one small server, would you **merge** each adapter or **keep them separate**? Explain your reasoning in one or two lines.

## 7. Saving & loading a model (and its tokenizer!)

To deploy a model you first **save** it to disk, then **load** it wherever it runs. Both models and tokenizers use `save_pretrained(folder)` to write and `from_pretrained(folder)` to read back.

The single most common mistake here: **forgetting the tokenizer**. A model only understands token *ids*; without the matching tokenizer your app can't turn text into those ids correctly. **Always save and ship the tokenizer alongside the model**, ideally in the same folder.

In [ ]:
import os

save_dir = "lead_classifier"

# Save BOTH the model and the tokenizer to the same folder.
clf_model.save_pretrained(save_dir)
clf_tokenizer.save_pretrained(save_dir)

print("Files saved in", save_dir, ":")
print(os.listdir(save_dir))
# You'll see config.json, model.safetensors (the weights),
# plus tokenizer files (tokenizer.json / vocab.txt / etc.).

**What this does:** `save_pretrained(save_dir)` writes the model's **config** (architecture + our `id2label` map) and **weights** (`model.safetensors`); saving the tokenizer to the *same* folder makes them travel together. Because we saved `id2label` in the config earlier, the reloaded model will already know the `hot`/`warm`/`cold` names.

In [ ]:
# Load the pair back from disk, as if on a fresh server.
reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_dir)
reloaded_tok = AutoTokenizer.from_pretrained(save_dir)
reloaded_model.to(device)
reloaded_model.eval()

# Sanity check: the labels survived the round-trip.
print("Reloaded id2label:", reloaded_model.config.id2label)

inputs = reloaded_tok("ready to buy now", return_tensors="pt").to(device)
with torch.no_grad():
    logits = reloaded_model(**inputs).logits
pred_id = torch.argmax(logits, dim=-1).item()
print("Reloaded prediction:", reloaded_model.config.id2label[pred_id])
print("Round-trip successful — model + tokenizer restored from disk.")

**What this does:** `from_pretrained(save_dir)` rebuilds the exact model and tokenizer you saved — same weights, same label map. The sanity check confirms inference still works and the labels came back intact. This round-trip is precisely what happens between **training** (one machine) and **serving** (another).

### ✏️ Exercise

Save the model + tokenizer to a *different* folder name (e.g. `"lead_classifier_backup"`), reload from there, and run `predict`-style inference on one sentence. Confirm you get a label back and that `reloaded_model.config.id2label` still shows the three lead labels.

In [ ]:
# Your turn:
# clf_model.save_pretrained("lead_classifier_backup")
# clf_tokenizer.save_pretrained("lead_classifier_backup")
# m = AutoModelForSequenceClassification.from_pretrained("lead_classifier_backup")
# print(m.config.id2label)

## 8. Batched inference for speed

Imagine 1,000 new leads arrive overnight. You *could* loop in Python and call the model 1,000 times — but that's slow, because each call has fixed overhead and the hardware sits half-idle. Much faster: **batch** them — tokenize the whole list at once with `padding=True`, then run the model **one time** on all of them.

Why is batching faster? GPUs and modern CPUs are built to do **many computations in parallel**. Feeding one example wastes most of that capacity; feeding a batch keeps the hardware busy. Same total math, far less overhead.

`padding=True` is required because the texts have different lengths — padding makes every row the same length so they fit in one rectangular tensor. The `attention_mask` tells the model which positions are real vs. padding.

In [ ]:
leads = [
    "Asked for a demo and pricing for 200 seats.",
    "Just browsing your blog.",
    "Wants to renew the annual contract this week.",
    "Downloaded a whitepaper, no reply since.",
    "Replied 'not interested, please remove me'.",
]

# Tokenize the WHOLE LIST at once. padding=True pads to the longest in the batch.
batch = clf_tokenizer(
    leads,
    return_tensors="pt",
    padding=True,        # make all rows equal length
    truncation=True,     # cut anything overly long
).to(device)

print("input_ids shape:", batch["input_ids"].shape)  # [5, seq_len] -> 5 rows at once

with torch.no_grad():
    logits = clf_model(**batch).logits   # ONE forward pass for all 5 leads

probs = torch.softmax(logits, dim=-1)
pred_ids = torch.argmax(probs, dim=-1)   # one prediction per row

for text, pid in zip(leads, pred_ids):
    print(f"{clf_model.config.id2label[pid.item()]:>5}  <-  {text}")

**What this does:** `clf_tokenizer(leads, padding=True, truncation=True)` tokenizes **all five** leads at once into a single `[5, seq_len]` tensor; `clf_model(**batch)` runs **one** forward pass for all five instead of five separate calls; `argmax(dim=-1)` gives one predicted id per row. For a few items the speed-up is small, but across thousands of leads batching can be **many times faster** than a Python loop.

### ✏️ Exercise

Build a list of **8** lead descriptions and run them through the batched path above. Print the `input_ids` shape (the first number should be `8`) and the predicted label for each. Then think: if you had 10,000 leads, why might you process them in batches of, say, 32 rather than all 10,000 at once? (Hint: memory.)

In [ ]:
# Your turn:
# my_leads = [ ... 8 strings ... ]
# batch = clf_tokenizer(my_leads, return_tensors="pt", padding=True, truncation=True).to(device)
# print(batch["input_ids"].shape)
# with torch.no_grad():
#     preds = torch.argmax(clf_model(**batch).logits, dim=-1)
# print([clf_model.config.id2label[p.item()] for p in preds])

## 9. Serving the model behind an API

So far we've called the model from inside the notebook. In a real product, your model runs as a **service**: a small web server that loads the model **once at startup**, then answers prediction requests over HTTP. Your app (a website, a CRM, a script) sends text to a `/predict` URL and gets back JSON like `{"label": "hot", "confidence": 0.82}`.

We'll use **FastAPI** (a popular, beginner-friendly Python web framework) plus **uvicorn** (the server that runs it). The cell below is a **complete app** — but it's shown as a **code block to copy**, *not* to run inside the notebook. You can't run a long-lived web server in a notebook cell (it would block forever). Instead:

1. Copy the code into a file named **`app.py`**.
2. Install the tools: `pip install fastapi uvicorn`.
3. Run it from a terminal: `uvicorn app:app --reload`.
4. Open `http://127.0.0.1:8000/docs` in a browser to try the `/predict` endpoint.

> **Do not run the next cell here.** It's a template. Notice it loads the model **once** at module load (startup), not on every request — loading per request would be painfully slow.

In [ ]:
# ===========================================================================
#  app.py  —  COPY THIS INTO A FILE CALLED app.py. DO NOT RUN IT IN THE NOTEBOOK.
#  Run it from a terminal with:  uvicorn app:app --reload
# ===========================================================================
#
# from fastapi import FastAPI
# from pydantic import BaseModel
# import torch
# from transformers import (
#     AutoTokenizer,
#     AutoModelForSequenceClassification,
# )
#
# # --- Load the model ONCE at startup (not per request!) ---
# MODEL_DIR = "lead_classifier"   # the folder we saved in section 7
# tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
# model.eval()
#
# app = FastAPI()
#
# # Describe the JSON the client must send: {"text": "..."}
# class Lead(BaseModel):
#     text: str
#
# @app.post("/predict")
# def predict(lead: Lead):
#     inputs = tokenizer(lead.text, return_tensors="pt")
#     with torch.no_grad():
#         logits = model(**inputs).logits
#     probs = torch.softmax(logits, dim=-1)
#     pred_id = torch.argmax(probs, dim=-1).item()
#     return {
#         "label": model.config.id2label[pred_id],
#         "confidence": round(probs[0, pred_id].item(), 3),
#     }
#
# # Optional health check so you can confirm the server is up:
# @app.get("/")
# def home():
#     return {"status": "ok"}

print("This is the app.py template — copy it to a file and run with uvicorn.")

**What this does:**

- **Load once at startup:** the `AutoTokenizer`/`AutoModel` lines run when `uvicorn` imports `app.py`, so the (slow) model load happens a single time. Each request then only does a fast forward pass.
- **`class Lead(BaseModel)`** declares the expected input JSON shape (`{"text": "..."}`). FastAPI validates it automatically.
- **`@app.post("/predict")`** defines the endpoint. Inside, it's the *same* tokenize → forward pass → softmax → argmax → label recipe from section 5, returned as JSON.
- The `/` health-check lets you (or a load balancer) verify the server is alive.

Once running, a client could `POST {"text": "wants a demo asap"}` to `/predict` and get `{"label": "...", "confidence": ...}` back. That's your fine-tuned lead-scorer, live in a product.

### Beyond a hand-rolled server

You won't always write your own FastAPI app. Common alternatives, depending on scale and budget:

- **Hugging Face Inference Endpoints** — upload your model to the Hub and HF hosts a managed API for you (no server code to maintain).
- **`text-generation-inference` (TGI)** — Hugging Face's optimized server for *generation* models, built for high throughput.
- **vLLM** — a very fast serving engine for large language models, popular for production LLM APIs with many concurrent users.

For a single small classifier like the capstone's, a tiny FastAPI app on a modest server is perfectly fine. Reach for TGI/vLLM/managed endpoints when you're serving large generation models at high traffic.

### ✏️ Exercise

You don't have to run a server. Instead, write down the **JSON request** you'd send to `/predict` for a hot lead, and the **JSON response** you'd expect back. Then name one reason loading the model *inside* the `predict` function (instead of at startup) would be a bad idea.

## 10. Making inference cheaper

Serving a model costs CPU/GPU time and memory, which costs money. A few beginner-friendly levers to make inference cheaper and faster — without retraining:

- **Quantization at inference time** — store the model's weights in lower precision (e.g. 8-bit or 4-bit instead of 32-bit floats). The model gets **much smaller in memory** and often faster, with usually a small accuracy hit. Libraries like `bitsandbytes` make this a one-line option when loading. (You met quantization in the QLoRA notebook for *training*; the same idea helps at *inference*.)
- **Use a smaller model** — a distilled or tiny model (like the ones in this notebook) is cheaper than a giant one. If a small fine-tuned model hits your accuracy target, prefer it. Bigger isn't always better.
- **Caching** — if the same inputs recur (e.g. duplicate lead descriptions, or repeated prompts), store recent results in a cache (even a simple Python dict or Redis) and return the saved answer instead of re-running the model.

Start simple: a small fine-tuned model, served once at startup, with a cache for repeats. Add quantization only if memory or speed becomes a real problem.

In [ ]:
# Tiny illustration of CACHING — skip the model when we've seen the text before.
_cache = {}

def predict_cached(text):
    if text in _cache:                 # already computed -> reuse it
        return _cache[text], "from cache"
    result = predict(text)             # otherwise run the model
    _cache[text] = result
    return result, "from model"

print(predict_cached("wants a demo asap"))  # first time -> from model
print(predict_cached("wants a demo asap"))  # second time -> from cache (instant)

**What this does:** `predict_cached` checks a dict first; if the exact text was seen before it returns the stored result **without running the model**. The second identical call is essentially free. Caching is the simplest of the three cost levers — quantization and smaller models need more setup, but a cache is often just a few lines.

### ✏️ Exercise

Extend `predict_cached` to also **count cache hits vs. misses** (add two counters and print them). Call it several times with a mix of repeated and new texts, then print how many times you hit the cache. This is a tiny version of the metrics a real service tracks.

In [ ]:
# Your turn:
# hits, misses = 0, 0
# def predict_cached_v2(text):
#     global hits, misses
#     ...

## Common mistakes & how to debug them

- **Forgetting `model.eval()` and/or `torch.no_grad()`.** Won't usually crash, but wastes memory and can change outputs (dropout stays on without `eval()`). Make both a reflex for inference.
- **Reading logits as probabilities.** Logits are raw scores and can be negative or large. Always apply `torch.softmax(logits, dim=-1)` before treating a number as a confidence.
- **Shipping the model without the tokenizer.** The model only understands token ids; without the matching tokenizer your text turns into the *wrong* ids and predictions are garbage. Always `save_pretrained` the tokenizer too, ideally in the same folder.
- **Padding errors with GPT-2.** Generation/batching complains about a missing pad token. Fix: `tokenizer.pad_token = tokenizer.eos_token`, and pass `pad_token_id=tokenizer.pad_token_id` to `generate`.
- **Tensors on the wrong device.** A `device mismatch` error means the model is on GPU but the inputs are on CPU (or vice-versa). Always `.to(device)` **both** the model and the tokenized inputs.
- **Expecting greedy output to vary (or sampled output to repeat).** Greedy (`do_sample=False`) is deterministic — same every run. Sampling (`do_sample=True`) changes each run unless you set `torch.manual_seed(...)`.
- **Loading the model inside the request handler.** In a server, this re-loads the model on *every* request and is brutally slow. Load it **once at startup** (module level), then only run the forward pass per request.
- **Trying to run the FastAPI server inside a notebook cell.** It blocks forever. Save it as `app.py` and run `uvicorn app:app` from a terminal.

## Summary

- **Inference = forward pass only.** Wrap it in `model.eval()` + `torch.no_grad()` for correct, lightweight predictions.
- **Text generation** with `model.generate(...)`: `max_new_tokens` caps length; `do_sample=False` is **greedy** (deterministic), `do_sample=True` is **sampling** (varied), tuned by `temperature`, `top_k`, `top_p`. Set `pad_token_id`/`eos_token_id` so generation stops and pads cleanly.
- **Classifier inference** is the **logits → softmax → argmax → label** chain, mapped through `id2label` to `hot`/`warm`/`cold`, wrapped in a reusable `predict(text)` helper.
- **LoRA at inference:** `PeftModel.from_pretrained(base_model, adapter_path)` attaches an adapter; `merge_and_unload()` folds it in for a standalone model. Merge for simple single-task deploys; keep separate to swap adapters.
- **Save & load** with `save_pretrained` / `from_pretrained` — and **always ship the tokenizer** alongside the model.
- **Batched inference** (`padding=True`, one forward pass on a list) is far faster than a Python loop.
- **Serving:** a minimal FastAPI `/predict` app that loads the model **once at startup**; scale up with TGI, vLLM, or managed HF endpoints. Cut cost with **quantization, smaller models, and caching**.

## What to learn next

Next up: **`15_mini_project_lead_intent.ipynb`** — the **capstone**. You'll put *everything* together: prepare the lead dataset, fine-tune a small model to classify leads as `hot` / `warm` / `cold`, evaluate it, save it, and run inference on brand-new leads. The model you train there is loaded and served using the exact techniques from this notebook — closing the loop from training to a working, queryable lead-scorer.